In [2]:
#imports and paths
import sys
import json
from pathlib import Path

#make the repository root importable from notebooks
root_candidates = []
if "__file__" in globals():
    root_candidates.append(Path(__file__).resolve().parent.parent)
root_candidates.extend([Path.cwd().resolve(), Path.cwd().resolve().parent])

for candidate in root_candidates:
    if (candidate / "retrieval").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
        break

from retrieval.pipeline import retrieve

with open("finetuning/eval_set.json", encoding="utf-8") as f:
    eval_set = json.load(f)

print(f"{len(eval_set)} held-out eval queries loaded")

c:\BA\advanced-rag-system\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


15 held-out eval queries loaded


In [3]:
#fetch retrieved context for each eval query, checkpointed
CHECKPOINT_PATH = Path("finetuning/eval_context_checkpoint.json")

def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
    return {}

def save_checkpoint(cp):
    CHECKPOINT_PATH.parent.mkdir(exist_ok=True)
    CHECKPOINT_PATH.write_text(json.dumps(cp, ensure_ascii=False, indent=2), encoding="utf-8")

checkpoint = load_checkpoint()

for i, q in enumerate(eval_set, 1):
    key = q["query"]
    if key in checkpoint:
        print(f"[{i}/{len(eval_set)}] SKIP: {key[:50]!r}")
        continue

    print(f"[{i}/{len(eval_set)}] {key[:50]!r}")
    chunks = retrieve(key, top_k=5, candidate_pool=10, history=[])

    checkpoint[key] = {
        "query": key,
        "language": q["language"],
        "ground_truth_chunk_id": q["answer_chunk_id"],
        "context_chunks": [
            {"chunk_id": cid, "text": chunk["text"], "source_doc": chunk.get("source_doc", cid)}
            for cid, chunk, score in chunks
        ],
        # was the ground-truth chunk actually retrieved at all? if not, no
        # model — base or fine-tuned — can possibly cite it correctly, and
        # that's a retrieval-stage fact worth keeping separate from generation quality
        "ground_truth_retrieved": q["answer_chunk_id"] in [cid for cid, _, _ in chunks],
    }
    save_checkpoint(checkpoint)

print(f"\nDone. {len(checkpoint)} queries with context prepared.")
retrieved_count = sum(1 for v in checkpoint.values() if v["ground_truth_retrieved"])
print(f"Ground truth chunk was retrieved for {retrieved_count}/{len(checkpoint)} queries")

[1/15] 'What is the definition of a sentence according to '
[bootstrap] loading persisted FAISS index...
[DEBUG] raw='What is the definition of a sentence according to the grammar text?' -> rewritten='What is the definition of a sentence according to the grammar text?'
[bm25] loading persisted index...
[vector_search] loading BAAI/bge-m3...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 19853.19it/s]


[rerank] loading BAAI/bge-reranker-base...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2069.72it/s]


[2/15] 'Who formed the Afrikander Ministry in October 1897'
[DEBUG] raw='Who formed the Afrikander Ministry in October 1897?' -> rewritten='Who formed the Afrikander Ministry in October 1897?'
[3/15] 'How did Mr. Cruncher recruit his finances?'
[DEBUG] raw='How did Mr. Cruncher recruit his finances?' -> rewritten='How did Mr. Cruncher acquire his finances?'
[4/15] 'How did GPUs change parallel computing according t'
[llm] Suspicious/empty output, retrying in 5s... (attempt 1/3): ''
[DEBUG] raw='How did GPUs change parallel computing according to the text?' -> rewritten='How did GPUs alter parallel computing as described in the text?'
[5/15] "What strange fancy grew in the narrator's mind abo"
[DEBUG] raw="What strange fancy grew in the narrator's mind about the house?" -> rewritten='What strange fancy did the narrator have about the house?'
[6/15] 'Why is C still popular despite newer programming l'
[DEBUG] raw='Why is C still popular despite newer programming languages?' -> rewritten=

In [4]:
# save final eval-context file
with open("finetuning/eval_context.json", "w", encoding="utf-8") as f:
    json.dump(list(checkpoint.values()), f, ensure_ascii=False, indent=2)

print("Saved finetuning/eval_context.json — upload this to the Colab evaluation notebook.")

Saved finetuning/eval_context.json — upload this to the Colab evaluation notebook.
